# Exploring the fruit fly connectome

A **connectome** is a complete wiring diagram of a nervous system: every neuron, and every connection between them.

This notebook explores **MaleCNS v1.0**, the connectome of one male fruit fly's entire central nervous system (brain + nerve cord). It was made by HHMI Janelia, the University of Cambridge, MRC LMB and Google Research, and is shared under a CC-BY 4.0 license.

**How the data was made, in five steps:**
1. The fly's nervous system was preserved, stained, and cut into layers far thinner than a hair.
2. Each layer was photographed with an **electron microscope**, sharp enough to see synapses (8 nanometers per pixel).
3. AI traced every cell through all the layers. Each traced 3D shape is called a **body**.
4. AI found every **synapse** (connection point) and guessed which chemical each neuron uses.
5. Scientists spent years checking the tracing and giving each neuron a name.

By the end of this notebook, we turn all of this into a network we can simulate.

## 1. Load the two tables

We load two files:

- **Annotations** (`ann`): a profile for each body. *What* is it? Its name, its category, which side of the body it's on.
- **Neurotransmitters** (`nt`): *how* each body talks to others. Neurons send signals using chemicals called neurotransmitters, and the chemical decides whether the signal says "fire!" or "be quiet!".

Feather is a fast file format for tables. `.shape` tells us (rows, columns).

### What is a body?

A body is any single traced 3D object in the image stack, whatever it turns out to be. The word was chosen because, at the moment of tracing, nobody knows yet what each object is. A body can be:

- A whole neuron, its cell body plus all its branches. After proofreading, each real neuron is ideally exactly one body. These are the 166,700 bodies with a superclass.
- A glial cell, a support cell that isn’t a neuron.
- A fragment, a piece of a neuron that got cut off during tracing, typically a thin branch the AI lost track of between image layers. Fragments get their own body IDs because, as far as the tracing knows, they’re separate objects.

In [1]:
import pandas as pd

ann = pd.read_feather("../data/body-annotations-male-cns-v1.0-minconf-0.5.feather") # annotations
nt = pd.read_feather("../data/body-neurotransmitters-male-cns-v1.0.feather") # neurotransmitters

print("Annotations:", ann.shape)
print(list(ann.columns))
print()
print("Neurotransmitters:", nt.shape)
print(list(nt.columns))

Annotations: (211577, 36)
['assignedOlHex1', 'assignedOlHex2', 'bodyId', 'flywireType', 'group', 'instance', 'somaSide', 'statusLabel', 'superclass', 'type', 'vfbId', 'hemibrainType', 'itoleeHl', 'supertype', 'birthtime', 'mancBodyid', 'mancGroup', 'mancType', 'subclass', 'synonyms', 'class', 'rootSide', 'somaNeuromere', 'trumanHl', 'dimorphism', 'matchingNotes', 'entryNerve', 'mancSerial', 'mcnsSerial', 'serialMotif', 'fruDsx', 'exitNerve', 'receptorType', 'somaLocation', 'tosomaLocation', 'status']

Neurotransmitters: (1835518, 10)
['body', 'cell_type', 'total_nt_predictions', 'predicted_nt_confidence', 'predicted_nt', 'ground_truth', 'celltype_total_nt_predictions', 'celltype_predicted_nt', 'celltype_predicted_nt_confidence', 'consensus_nt']


### First look at the annotations

Each row is one body. The very first row is famous: `DNp01(GF)_R` is the **Giant Fiber**, the neuron that makes a fly jump away when something swoops at it. It is one of the fastest escape pathways known in any animal.

Notice how much is empty (`NaN` / `None`). Most columns only apply to some neurons, so empty cells are normal.

In [2]:
ann.head()

,assignedOlHex1,assignedOlHex2,bodyId,flywireType,group,instance,somaSide,statusLabel,superclass,type,...,entryNerve,mancSerial,mcnsSerial,serialMotif,fruDsx,exitNerve,receptorType,somaLocation,tosomaLocation,status
0,NaN,NaN,10001,DNp01,10001.0,DNp01(GF)_R,R,Roughly traced,descending_neuron,DNp01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[37124, 22258, 36274]",None,Traced
1,NaN,NaN,10002,OCG01d,10002.0,OCG01d_L,L,Roughly traced,visual_projection,OCG01d,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[50214, 10279, 27497]",None,Traced
2,NaN,NaN,10003,VCH,10003.0,VCH_R,R,Prelim Roughly traced,visual_centrifugal,VCH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[46716, 21029, 15714]",None,Traced
3,NaN,NaN,10005,AOTU019,10005.0,AOTU019_R,R,Roughly traced,cb_intrinsic,AOTU019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[41190, 18491, 12885]",None,Traced
4,NaN,NaN,10006,"VS1,VS2,VS3,VS4,VS5,VS6,VS7,VS8",10006.0,VS_L,L,Prelim Roughly traced,visual_projection,VS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[69604, 28368, 42459]",None,Traced


### First look at the neurotransmitters

**How can a computer guess a neuron's chemical from a photo?**
At each synapse, the sending neuron has a dark "T"-shaped structure, the **T-bar**, where tiny packets of neurotransmitter wait to be released. Different chemicals come with slightly different-looking packets and T-bars. An AI learned these differences, guesses the chemical at each T-bar, then combines the guesses for the whole neuron.

Things to notice:
- `total_nt_predictions` = how many T-bars were used. More T-bars = a more trustworthy vote.
- `ground_truth` = the chemical proven by real lab experiments. Rare, but the most reliable.
- The Giant Fiber's confidence is only **0.50**. Its most important connection (to the jump muscle) is an **electrical synapse**: a direct channel between cells, with no T-bars. Electrical synapses are invisible in this data.

In [3]:
nt.head()

,body,cell_type,total_nt_predictions,predicted_nt_confidence,predicted_nt,ground_truth,celltype_total_nt_predictions,celltype_predicted_nt,celltype_predicted_nt_confidence,consensus_nt
0,10001,DNp01,1015,0.500496,acetylcholine,NaN,1991,acetylcholine,0.528263,acetylcholine
1,10002,OCG01d,1443,0.955895,acetylcholine,NaN,2818,acetylcholine,0.956476,acetylcholine
2,10003,VCH,4425,0.879325,gaba,gaba,8721,gaba,0.874799,gaba
3,10005,AOTU019,2837,0.835076,gaba,gaba,5673,gaba,0.830504,gaba
4,10006,VS,426,0.904134,acetylcholine,acetylcholine,5260,acetylcholine,0.820038,acetylcholine


## 2. Do the two tables match up?

Both tables identify bodies by number (`bodyId` in one, `body` in the other). We check:

- Each ID appears only once (`is_unique`), so one ID = one body.
- How many IDs appear in **both** tables. Those are the bodies we know both *what* they are and *how* they signal.

Why don't they match perfectly? A transmitter guess needs T-bars. Bodies with **no** T-bars (like glia, the brain's support cells, or small fragments) get no guess at all. And the transmitter table is bigger because it includes millions of tiny fragments.

In [4]:
ann_id = "bodyId"   # ID column name in the annotations table
nt_id = "body"      # ID column name in the neurotransmitters table

print("Annotation IDs unique?", ann[ann_id].is_unique)
print("Neurotransmitter IDs unique?", nt[nt_id].is_unique)

shared = set(ann[ann_id]) & set(nt[nt_id])
print("IDs in both files:", len(shared))
print("Annotated neurons with no transmitter prediction:", len(set(ann[ann_id]) - shared))

Annotation IDs unique? True
Neurotransmitter IDs unique? True
IDs in both files: 187016
Annotated neurons with no transmitter prediction: 24561


## 3. How the nervous system is organized

`superclass` sorts neurons into big families. Together they show how information flows:

**senses → processing → commands → muscles**

| Superclass | What it means |
|---|---|
| `ol_sensory` | Light sensors in the eyes (photoreceptors) |
| `cb_sensory`, `vnc_sensory` | Senses of the head (smell, taste) and body (touch, leg position) |
| `ol_intrinsic` | Neurons inside the optic lobes that process vision |
| `cb_intrinsic` | Neurons inside the central brain: learning, navigation, decisions |
| `vnc_intrinsic` | Neurons inside the nerve cord that coordinate legs and wings |
| `descending_neuron` | Commands sent **down** from brain to body |
| `ascending_neuron` | Reports sent **up** from body to brain |
| `vnc_motor`, `cb_motor` | Neurons that move muscles |
| `None` | Not a neuron (glia, fragments). We will drop these. |

Big surprise: **more than half** of all neurons (89,403) are in the optic lobes. Flies are very visual animals.

In [5]:
ann["superclass"].value_counts(dropna=False)

superclass
ol_intrinsic             89403
NaN                      44877
cb_intrinsic             32164
vnc_intrinsic            13161
visual_projection         9201
vnc_sensory               6370
ol_sensory                6098
cb_sensory                4868
ascending_neuron          1846
descending_neuron         1314
vnc_motor                  708
visual_centrifugal         563
sensory_ascending          537
cb_motor                   107
vnc_efferent                94
cb_endocrine                72
ENS                         50
vnc_tbc                     38
vnc_sensory_tbc             36
vnc_endocrine               22
cb_sensory_tbc              14
sensory_descending          12
efferent_ascending           8
efferent_descending          4
cb_efferent                  4
visual_projection_tbc        2
sensory_ascending_tbc        2
descending_neuron_tbc        2
Name: count, dtype: int64

## 4. Find one specific neuron type: DNa02

Neurons with the same shape and connections share a **type** name. Most types come in left/right pairs.

**DNa02** is a descending neuron involved in **steering** while walking: when one side fires more than the other, the fly turns. There are exactly two: one on the left (`_L`) and one on the right (`_R`).

In [6]:
dna02 = ann[ann["type"] == "DNa02"]
dna02

,assignedOlHex1,assignedOlHex2,bodyId,flywireType,group,instance,somaSide,statusLabel,superclass,type,...,entryNerve,mancSerial,mcnsSerial,serialMotif,fruDsx,exitNerve,receptorType,somaLocation,tosomaLocation,status
337,NaN,NaN,10360,DNa02,10360.0,DNa02_R,R,Prelim Roughly traced,descending_neuron,DNa02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[33699, 18906, 15758]",None,Traced
155705,NaN,NaN,523769,DNa02,10360.0,DNa02_L,L,Prelim Roughly traced,descending_neuron,DNa02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[61859, 18876, 14052]",None,Traced


## 5. Which column holds the neurotransmitter?

We print the most common values of every text column to find the chemical names.

Notice how many say `unclear`: those are mostly tiny fragments with too few T-bars to judge.

There are several transmitter columns:
- `predicted_nt`: the guess for one body
- `celltype_predicted_nt`: the guess after pooling **all** neurons of the same type (more reliable, because neurons of one type almost always use the same chemical)
- `consensus_nt`: the final recommended answer. **This is the one we use.**

In [7]:
for col in nt.columns:
    if nt[col].dtype == "object" or str(nt[col].dtype) == "string":
        print(f"--- {col} ---")
        print(nt[col].value_counts().head(10))
        print()

## 6. The descending neurons

Every command from the brain to the body travels through this group, only about **1,300 neurons** out of 166,700. It's a narrow highway, which makes these neurons natural "output buttons" for a simulation.

`str.contains("descending")` also catches related labels like `sensory_descending`, so the count is slightly higher than the `descending_neuron` superclass alone.

In [8]:
desc = ann[ann["superclass"].str.contains("descending", case=False, na=False)]
print(len(desc), "descending neurons")
desc["type"].value_counts().head(20)

1332 descending neurons


type
DNg08      23
DNpe008    19
DNg106     16
DNg07      16
DNge091    14
DNge094    13
DNg03      12
DNg06      12
DNp17      12
DNg10      12
DNge019    11
DNpe015    11
DNg02_a    10
DNg12_b    10
DNge071    10
DNg12_a     8
DNg12_c     8
DNge115     8
DNge085     8
DNg36_b     7
Name: count, dtype: int64

## 7. Describe every column

This helper prints, for each column: its data type, how full it is (%), and a few example values. It's a quick way to understand an unfamiliar table.

A column that is only 2% filled usually describes a small group of neurons (for example `entryNerve` only applies to sensory neurons).

**Why `.astype(str)`?** Some columns, like `somaLocation`, hold little lists of 3 numbers (x, y, z). Finding unique lists is very slow, which froze the notebook once. Turning everything into text first avoids that.

In [9]:
def describe_table(df, name):
    print(f"===== {name}: {df.shape[0]:,} rows, {df.shape[1]} columns =====", flush=True)
    for col in df.columns:
        print(f"{col:30} ", end="", flush=True)
        filled = df[col].notna().sum()
        pct = 100 * filled / len(df)
        try:
            examples = df[col].dropna().astype(str).unique()[:5]
        except Exception as e:
            examples = [f"(could not read examples: {type(e).__name__})"]
        examples = [ex[:40] for ex in examples]
        print(f"{str(df[col].dtype):10} {pct:5.1f}% filled   e.g. {examples}", flush=True)
    print()

describe_table(ann, "Annotations")

===== Annotations: 211,577 rows, 36 columns =====
assignedOlHex1                 float64     11.2% filled   e.g. ['12.0', '28.0', '5.0', '17.0', '20.0']
assignedOlHex2                 float64     11.2% filled   e.g. ['26.0', '22.0', '9.0', '28.0', '12.0']
bodyId                         int64      100.0% filled   e.g. ['10001', '10002', '10003', '10005', '10006']
flywireType                    str         67.7% filled   e.g. ['DNp01', 'OCG01d', 'VCH', 'AOTU019', 'VS1,VS2,VS3,VS4,VS5,VS6,VS7,VS8']
group                          float64     69.7% filled   e.g. ['10001.0', '10002.0', '10003.0', '10005.0', '10006.0']
instance                       str         76.3% filled   e.g. ['DNp01(GF)_R', 'OCG01d_L', 'VCH_R', 'AOTU019_R', 'VS_L']
somaSide                       str         71.2% filled   e.g. ['R', 'L', 'M']
statusLabel                    category    99.6% filled   e.g. ['Roughly traced', 'Prelim Roughly traced', 'Unimportant', 'Reviewed', 'Glia']
superclass                     str    

In [10]:
describe_table(nt, "Neurotransmitters")

===== Neurotransmitters: 1,835,518 rows, 10 columns =====
body                           int64      100.0% filled   e.g. ['10001', '10002', '10003', '10005', '10006']
cell_type                      str          9.0% filled   e.g. ['DNp01', 'OCG01d', 'VCH', 'AOTU019', 'VS']
total_nt_predictions           int32      100.0% filled   e.g. ['1015', '1443', '4425', '2837', '426']
predicted_nt_confidence        float64    100.0% filled   e.g. ['0.5004955542656235', '0.9558953111126071', '0.8793247902745723', '0.8350762055239446', '0.9041343101711969']
predicted_nt                   str        100.0% filled   e.g. ['acetylcholine', 'gaba', 'octopamine', 'glutamate', 'unclear']
ground_truth                   str          4.7% filled   e.g. ['gaba', 'acetylcholine', 'glutamate', 'octopamine', 'serotonin']
celltype_total_nt_predictions  int32      100.0% filled   e.g. ['1991', '2818', '8721', '5673', '5260']
celltype_predicted_nt          str        100.0% filled   e.g. ['acetylcholine', 'gaba', 

### Superclass counts again

Same as section 3. A neat check: add up every superclass except `None` and you get **exactly 166,700**, the published number of neurons. So "has a superclass" is a clean way to tell neurons apart from everything else.

In [11]:
ann["superclass"].value_counts(dropna=False)

superclass
ol_intrinsic             89403
NaN                      44877
cb_intrinsic             32164
vnc_intrinsic            13161
visual_projection         9201
vnc_sensory               6370
ol_sensory                6098
cb_sensory                4868
ascending_neuron          1846
descending_neuron         1314
vnc_motor                  708
visual_centrifugal         563
sensory_ascending          537
cb_motor                   107
vnc_efferent                94
cb_endocrine                72
ENS                         50
vnc_tbc                     38
vnc_sensory_tbc             36
vnc_endocrine               22
cb_sensory_tbc              14
sensory_descending          12
efferent_ascending           8
efferent_descending          4
cb_efferent                  4
visual_projection_tbc        2
sensory_ascending_tbc        2
descending_neuron_tbc        2
Name: count, dtype: int64

## 8. Build the neuron list

Two steps:
1. **Keep only real neurons**: rows that have a superclass.
2. **Attach each neuron's chemical**: `merge` works like a lookup, matching `bodyId` to `body`. `how="left"` keeps every neuron even if it has no transmitter guess, and those get the label `unknown`.

Result: about 62% of neurons use **acetylcholine** (the main "go" signal in flies), and about 30% use **GABA** or **glutamate** (mostly "stop" signals).

In [12]:
import numpy as np

# 1. Keep only real neurons
real = ann[ann["superclass"].notna()].copy()
print("Real neurons:", len(real))

# 2. Attach each neuron's transmitter
nt_small = nt[["body", "consensus_nt"]]
neurons = real.merge(nt_small, left_on="bodyId", right_on="body", how="left")
neurons = neurons.drop(columns=["body"])
neurons["consensus_nt"] = neurons["consensus_nt"].fillna("unknown")
print(neurons["consensus_nt"].value_counts())

Real neurons: 166700
consensus_nt
acetylcholine    103720
glutamate         29302
gaba              22069
histamine          7891
unclear            2999
dopamine            392
unknown             178
octopamine          101
serotonin            48
Name: count, dtype: int64


## 9. Excite or inhibit?

Each neuron gets a **sign**:
- **+1 (excitatory)**: its signal pushes the next neuron *toward* firing. Like pressing the accelerator.
- **-1 (inhibitory)**: its signal pushes the next neuron *away* from firing. Like pressing the brake.

In flies: acetylcholine excites, while GABA and glutamate usually inhibit. (In humans, glutamate is the main *excitatory* chemical, so this is a fly-specific rule.)

A neuron usually uses one chemical for all its outputs, so the sign belongs to the **sender**.

Simplification: dopamine, serotonin and octopamine actually work slowly, adjusting other circuits over seconds. Here we just treat them as +1.

In [13]:
# 3. Decide whether each neuron excites (+1) or inhibits (-1)
inhibitory = ["gaba", "glutamate"]
neurons["sign"] = np.where(neurons["consensus_nt"].isin(inhibitory), -1, 1)

print(neurons["sign"].value_counts())
print("Neuron count check:", len(neurons), "(should be 166,700)")

sign
 1    115329
-1     51371
Name: count, dtype: int64
Neuron count check: 166700 (should be 166,700)


### Fix: histamine is inhibitory too

**Histamine** is used mainly by the light sensors in the eyes. In flies, when light hits a photoreceptor, its histamine makes the next neurons **quieter**, not louder. Vision in flies starts with this "sign flip".

So we add histamine to the inhibitory list. Running this cell overwrites the `sign` column from before.

In [15]:
inhibitory = ["gaba", "glutamate", "histamine"]
neurons["sign"] = np.where(neurons["consensus_nt"].isin(inhibitory), -1, 1)
print(neurons["sign"].value_counts())

sign
 1    107438
-1     59262
Name: count, dtype: int64


### Free some memory

The transmitter table has 1.8 million rows, and we've already copied what we need. `del` removes it from memory (the file on disk is untouched).

In [16]:
del nt, nt_small, real

## 10. Peek inside the connections file

This file lists every connection: who sends (`body_pre`), who receives (`body_post`), and the **weight**, the number of synapses between them. More synapses = a stronger connection.

The file is huge (152 million rows), so instead of loading it all, we open it and read only its description and its first chunk. The file is stored as 2,318 chunks, which lets us process it piece by piece later.

The rows are sorted strongest first: the top pair is linked by **2,591 synapses**.

In [17]:
import pyarrow as pa

path = "../data/connectome-weights-male-cns-v1.0-minconf-0.5.feather"
reader = pa.ipc.open_file(path)

print(reader.schema)
print("Chunks in the file:", reader.num_record_batches)

first = reader.get_batch(0).to_pandas()
print("Rows in first chunk:", len(first))
first.head()

body_pre: int64
body_post: int64
weight: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 604
Chunks in the file: 2318
Rows in first chunk: 65536


,body_pre,body_post,weight
0,10352,10351,2591
1,13612,16076,2443
2,10663,10051,2248
3,10289,10611,2230
4,10893,10066,1929


## 11. Who is the strongest pair?

Both are **optic lobe** neurons in the **lobula**, a region that detects visual features like moving objects. The sender, `Li32`, uses **GABA**, so the strongest connection in the whole nervous system is a powerful **brake**: Li32 strongly silences Li33.

In [18]:
neurons[neurons["bodyId"].isin([10352, 10351])][["bodyId", "type", "instance", "superclass", "consensus_nt"]]

,bodyId,type,instance,superclass,consensus_nt
324,10351,Li33,Li33_R,ol_intrinsic,acetylcholine
325,10352,Li32,Li32_R,ol_intrinsic,gaba


## 12. Give every neuron a number from 0 to 166,699

Body IDs are big scattered numbers (like 10352 or 530880). A simulation works much faster if neurons are simply numbered 0, 1, 2, ... so neuron number *i* is row *i* of our table.

`pd.Index` turns the IDs into a fast lookup table: give it a body ID, get back its row number.

In [19]:
neurons = neurons.reset_index(drop=True)
neuron_ids = pd.Index(neurons["bodyId"])
print("IDs unique:", neuron_ids.is_unique)

IDs unique: True


## 13. Read the whole file, keeping only real connections

We read the 152 million rows one chunk at a time (so memory stays low) and keep a connection only if **both** ends are real neurons.

`get_indexer` converts body IDs into row numbers, and returns **-1** for anything that isn't one of our neurons (a fragment). So `keep` = "both ends are not -1".

**Result:** only 17% of rows survive (25.6 million connections), yet they carry 124 million synapses, almost all of them. The fragments were tiny scraps.
These numbers exactly match other published projects using this data, a good sign that everything so far is correct.

In [20]:
pre_list, post_list, weight_list = [], [], []
rows_seen = 0

for i in range(reader.num_record_batches):
    batch = reader.get_batch(i)
    pre = batch.column("body_pre").to_numpy()
    post = batch.column("body_post").to_numpy()
    weight = batch.column("weight").to_numpy()
    rows_seen += len(pre)

    pre_idx = neuron_ids.get_indexer(pre)
    post_idx = neuron_ids.get_indexer(post)
    keep = (pre_idx >= 0) & (post_idx >= 0)

    pre_list.append(pre_idx[keep].astype(np.int32))
    post_list.append(post_idx[keep].astype(np.int32))
    weight_list.append(weight[keep].astype(np.int32))

    if i % 200 == 0:
        print(f"chunk {i}/{reader.num_record_batches}, rows scanned: {rows_seen:,}", flush=True)

pre_all = np.concatenate(pre_list)
post_all = np.concatenate(post_list)
weight_all = np.concatenate(weight_list)
del pre_list, post_list, weight_list

print(f"Scanned {rows_seen:,} rows")
print(f"Kept {len(pre_all):,} connections between real neurons")
print(f"Total synapses: {weight_all.sum():,}")

chunk 0/2318, rows scanned: 65,536
chunk 200/2318, rows scanned: 13,172,736
chunk 400/2318, rows scanned: 26,279,936
chunk 600/2318, rows scanned: 39,387,136
chunk 800/2318, rows scanned: 52,494,336
chunk 1000/2318, rows scanned: 65,601,536
chunk 1200/2318, rows scanned: 78,708,736
chunk 1400/2318, rows scanned: 91,815,936
chunk 1600/2318, rows scanned: 104,923,136
chunk 1800/2318, rows scanned: 118,030,336
chunk 2000/2318, rows scanned: 131,137,536
chunk 2200/2318, rows scanned: 144,244,736
Scanned 151,856,684 rows
Kept 25,582,938 connections between real neurons
Total synapses: 124,177,617


## 14. Save the results

Saving the clean connections and neuron list means we never have to scan the big file again: loading these takes seconds.
We only save useful neuron columns (skipping list-type columns like `somaLocation`).

In [21]:
edges = pd.DataFrame({"pre": pre_all, "post": post_all, "weight": weight_all})
edges.to_parquet("../data/edges.parquet")

keep_cols = ["bodyId", "type", "instance", "superclass", "class",
             "somaSide", "rootSide", "consensus_nt", "sign"]
neurons[keep_cols].to_parquet("../data/neurons.parquet")
print("Saved!")

Saved!


## 15. Turn the connections into a network matrix

Imagine a giant grid: **row = sender, column = receiver**, and each cell holds the connection strength.
- Strength = number of synapses × the sender's sign. So 50 synapses from a GABA neuron become **-50**.

A full grid would have 166,700 × 166,700 ≈ 27.8 billion cells (about 111 GB!), but 99.9% would be zero, because each neuron connects to only ~150 others.
A **sparse matrix** stores only the non-zero cells: about 300 MB.

**CSR** format groups each row together, so "who does neuron *i* talk to?" is fast to answer. That's exactly the question a simulation asks every time a neuron fires.

In [22]:
import scipy.sparse as sp

n = len(neurons)
signs = neurons["sign"].to_numpy()
signed_weight = weight_all * signs[pre_all]

W = sp.coo_matrix((signed_weight, (pre_all, post_all)), shape=(n, n)).tocsr()
W.sum_duplicates()

print("Matrix shape:", W.shape)
print("Stored connections:", W.nnz)
print(f"Memory: {(W.data.nbytes + W.indices.nbytes + W.indptr.nbytes) / 1e6:.0f} MB")

sp.save_npz("../data/W.npz", W)
print("Saved!")

Matrix shape: (166700, 166700)
Stored connections: 25582938
Memory: 308 MB
Saved!


## 16. Who are the biggest broadcasters?

For each neuron, we add up all its outgoing synapses (ignoring sign) and list the top 15.

Almost all are **inhibitory (GABA)**: the nervous system's biggest broadcasters are its **brakes**.
- **APL** (one per side) covers the whole **mushroom body**, the fly's learning center. It quiets all the smell-coding neurons there, so only a few stay active for each smell. That keeps every smell's pattern distinct and easy to remember.
- **CT1** (one per side) spreads through the optic lobe, giving local feedback in hundreds of visual columns at once.

Without strong brakes like these, activity would spread out of control. (We saw exactly that happen later, in simulation!)

In [23]:
out_synapses = np.asarray(abs(W).sum(axis=1)).ravel()
neurons["out_synapses"] = out_synapses
neurons.sort_values("out_synapses", ascending=False)[
    ["bodyId", "type", "superclass", "consensus_nt", "out_synapses"]
].head(15)

,bodyId,type,superclass,consensus_nt,out_synapses
5,10009,CT1,ol_intrinsic,gaba,140897
141,10157,CT1,ol_intrinsic,gaba,112796
904,10977,APL,cb_intrinsic,gaba,107649
502,10540,APL,cb_intrinsic,gaba,102263
42,10051,Li39,ol_intrinsic,gaba,53934
53,10066,LPi21,ol_intrinsic,gaba,50226
569,10611,Li39,ol_intrinsic,gaba,49925
80,10093,Am1,ol_intrinsic,gaba,49864
466,10503,LPi4b,ol_intrinsic,gaba,47667
37,10046,MeVC11,visual_centrifugal,acetylcholine,46425


## Summary

What we built in this notebook:
1. A list of **166,700 neurons**, each with a type, a family, a chemical, and a sign (+1 / -1).
2. **25.6 million connections** carrying **124 million synapses**.
3. A **signed sparse matrix** `W`, saved to `data/W.npz`, ready for simulation.

Next notebook: making neurons fire, starting with a single neuron.